## Setup Libraries

In [1]:
import os

os.environ['USER'] = 'root'
!pip install xlearn -q

import xlearn as xl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 12.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os
import random
import warnings
import numpy as np, pandas as pd
from colorama import Fore, Style
from importlib.metadata import version
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import xlearn as xl

from tqdm import tqdm
import gc

tqdm.pandas(desc="Progress", leave=True, position=0, ncols=80)
warnings.filterwarnings('ignore')

In [3]:
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
seed_everything(seed=42)

## Load Data

In [4]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (594194, 21)
Test shape : (254655, 20)


## Preprocess Features

In [5]:
%%time
ID = 'id'
TARGET = 'Churn'
train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
del train, test
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
important_combos = [
    ('Contract', 'InternetService', 'PaymentMethod'),
]
def feature_engineering(df, fit=False):
    # Arithmetic interaction
    df['_MonthlyCharges_/_TotalCharges'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1e-6)).astype('float32')
    df['_TotalCharges_/_tenure']  = (df['TotalCharges'] / (df['tenure'] + 1e-6)).astype('float32')
    df['_Monthly_to_avg_ratio'] = (df['MonthlyCharges'] / (df['_TotalCharges_/_tenure'] + 1e-6)).astype('float32')
    df['_TotalCharges_/_MonthlyCharges']  = (df['TotalCharges'] / (df['MonthlyCharges'] + 1e-6)).astype('float32')
    df['_tenure_sq'] = (df['tenure'] ** 2).astype("float32")
    
    # Digit extraction
    for col in ['TotalCharges']:
        k = -3
        digit_name = f"{col}_d{k}_"
        df[digit_name] = ((df[col] * 10**k) % 10).astype('int8')
        df[digit_name] = df[digit_name].astype('category')

    # Discretize numericals
    bin_config = {'TotalCharges': [4000, 500], 'MonthlyCharges': [200, 100]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_bin_"
            if fit:
                kb = KBinsDiscretizer(
                    n_bins=n_bins,
                    encode='ordinal',
                    strategy='quantile',
                    subsample=None
                )
                binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                category_map[bin_name] = kb
            else:
                kb = category_map[bin_name]
                binned = kb.transform(df[[col]]).ravel().astype('int32')
            df[bin_name] = binned
            df[bin_name] = df[bin_name].astype('category')

    # Categorize numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        round_level = 0
        if fit:
            round_flag = col == 'TotalCharges'
            series = df[col].round(round_level) if round_flag else df[col]
            codes, uniques = series.factorize()
            category_map[col] = {'uniques': uniques, 'round_flag': round_flag}
        else:
            round_flag = category_map[col]['round_flag']
            uniques = category_map[col]['uniques']
            series = df[col].round(round_level) if round_flag else df[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = series.map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes    
        df[cat_name] = df[cat_name].astype('category')

    # Create interaction categories
    combo_names = []
    for col1, col2, col3 in important_combos:
        combo_name = f"{col1}_{col2}_{col3}_"
        combo_names.append(combo_name)
        combo_series = df[col1].astype(str) + '_' + df[col2].astype(str) + '_' + df[col3].astype(str)
        if fit:
            codes, uniques = combo_series.factorize()
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, new_cat_cols, new_num_cols, combo_names = feature_engineering(X_test, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape, "\n")

X      init shape: (594194, 19)
X_test init shape: (254655, 19) 

init len(cat_cols): 15
init len(num_cols): 4 

len(new_cat_cols): 10
len(new_num_cols): 5 

prep len(cat_cols): 25
prep len(num_cols): 9 

X      prep shape: (594194, 34)
X_test prep shape: (254655, 34) 

CPU times: user 1.08 s, sys: 209 ms, total: 1.29 s
Wall time: 1.32 s


In [6]:
class LibFFMEncoder(object):
    def __init__(self):
        self.encoder = 1
        self.encoding = {}
    
    def encode_for_libffm(self, row):
        # Use list concatenation instead of repeated string concatenation
        parts = [str(row[0])]
        
        for i, r in enumerate(row[1:]):
            key = (i, r)
            if key not in self.encoding:
                self.encoding[key] = self.encoder
                self.encoder += 1
            parts.append(f'{i}:{self.encoding[key]}:1')
        
        return ' '.join(parts)

## Config

In [7]:
class CFG:
    FOLDS = 5
    SEED = 42


## Train K-Fold

In [8]:
%%time
skf = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print("#"*16)
    print(f"### Fold {fold}/{CFG.FOLDS} ...")
    print("#"*16)

    X_tr = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    X_tst = X_test.copy()

    # Target encoding
    te_cols = combo_names
    TE = TargetEncoder(cv=CFG.FOLDS, smooth='auto', shuffle=True, random_state=CFG.SEED)
    tr_enc = TE.fit_transform(X_tr[te_cols], y.iloc[tr_idx])
    val_enc = TE.transform(X_val[te_cols])
    tst_enc = TE.transform(X_tst[te_cols])

    te_names = [f"_{col}TE" for col in te_cols]
    X_tr[te_names] = tr_enc
    X_val[te_names] = val_enc
    X_tst[te_names] = tst_enc

    X_test_dummy = X_tst.copy()
    X_test_dummy[TARGET] = 0
    y_test = X_test_dummy[TARGET]

    encoder = LibFFMEncoder()
    df_all = pd.concat([
        pd.concat([y.iloc[tr_idx], X_tr], axis=1),
        pd.concat([y.iloc[val_idx], X_val], axis=1),
        pd.concat([y_test, X_tst], axis=1),
    ], axis=0, ignore_index=True)

    _ = df_all.progress_apply(lambda row: encoder.encode_for_libffm(row), axis=1)

    libffm_format_train = pd.concat([y.iloc[tr_idx], X_tr], axis=1).progress_apply(
        lambda row: encoder.encode_for_libffm(row), raw=True, axis=1
    )

    libffm_format_val = pd.concat([y.iloc[val_idx], X_val], axis=1).progress_apply(
        lambda row: encoder.encode_for_libffm(row), raw=True, axis=1
    )
    
    libffm_format_test = pd.concat([y_test, X_tst], axis=1).progress_apply(
        lambda row: encoder.encode_for_libffm(row), raw=True, axis=1
    )

    libffm_format_train.to_csv(f'libffm_train_fold_{fold + 1}.txt', index=False, header=False)
    libffm_format_val.to_csv(f'libffm_val_fold_{fold + 1}.txt', index=False, header=False)
    libffm_format_test.to_csv(f'libffm_test_fold_{fold + 1}.txt', index=False, header=False)
    
    ffm = xl.create_ffm()
    ffm.setTrain(f'libffm_train_fold_{fold + 1}.txt')
    ffm.setValidate(f'libffm_val_fold_{fold + 1}.txt')
    ffm.fit({"task": "binary", "metric": "auc", 'nthread': 2, 'epoch': 50, 'lr': 0.05, 'lambda': 0.0001}, "model.out")
    ffm.setTest(f'libffm_val_fold_{fold + 1}.txt')
    ffm.predict("./model.out", "./output.txt")
    oof_preds[val_idx] = pd.read_csv("./output.txt", header=None).values.flatten()
    ffm.setTest(f'libffm_test_fold_{fold + 1}.txt')
    ffm.predict("./model.out", "./output.txt")
    test_preds += pd.read_csv("./output.txt", header=None).values.flatten()

    
    fold_score = roc_auc_score(y.iloc[val_idx], oof_preds[val_idx])
    print(f"{Fore.CYAN}Fold {fold} | XLEARN FFM AUC: {fold_score:.5f}{Style.RESET_ALL}\n")

    del X_tr, X_val
    gc.collect()

test_preds /= 5
oof_score = roc_auc_score(y, oof_preds)
print("\n" + "="*24)
print(f"Overall OOF AUC: {Fore.BLACK}{Style.BRIGHT}{oof_score:.5f}{Style.RESET_ALL}")
print("="*24)

################
### Fold 1/5 ...
################


Progress: 100%|██████████████████████| 254655/254655 [00:06<00:00, 37450.20it/s]


----------------------------------------------------------------------------------------------
           _
          | |
     __  _| |     ___  __ _ _ __ _ __
     \ \/ / |    / _ \/ _` | '__| '_ \ 
      >  <| |___|  __/ (_| | |  | | | |
     /_/\_\_____/\___|\__,_|_|  |_| |_|

        xLearn   -- 0.40 Version --
----------------------------------------------------------------------------------------------

[------------] xLearn uses 2 threads for training task.
[ ACTION     ] Read Problem ...
[------------] First check if the text file has been already converted to binary format.
[------------] Binary file (libffm_train_fold_2.txt.bin) NOT found. Convert text file to binary file.
[------------] First check if the text file has been already converted to binary format.
[------------] Binary file (libffm_val_fold_2.txt.bin) NOT found. Convert text file to binary file.
[------------] Number of Feature: 1548962
[------------] Number of Field: 35
[------------] Time cost for reading probl

Progress: 100%|██████████████████████| 254655/254655 [00:06<00:00, 37214.55it/s]


----------------------------------------------------------------------------------------------
           _
          | |
     __  _| |     ___  __ _ _ __ _ __
     \ \/ / |    / _ \/ _` | '__| '_ \ 
      >  <| |___|  __/ (_| | |  | | | |
     /_/\_\_____/\___|\__,_|_|  |_| |_|

        xLearn   -- 0.40 Version --
----------------------------------------------------------------------------------------------

[------------] xLearn uses 2 threads for training task.
[ ACTION     ] Read Problem ...
[------------] First check if the text file has been already converted to binary format.
[------------] Binary file (libffm_train_fold_3.txt.bin) NOT found. Convert text file to binary file.
[------------] First check if the text file has been already converted to binary format.
[------------] Binary file (libffm_val_fold_3.txt.bin) NOT found. Convert text file to binary file.
[------------] Number of Feature: 1548962
[------------] Number of Field: 35
[------------] Time cost for reading probl

...

----------------------------------------------------------------------------------------------
           _
          | |
     __  _| |     ___  __ _ _ __ _ __
     \ \/ / |    / _ \/ _` | '__| '_ \ 
      >  <| |___|  __/ (_| | |  | | | |
     /_/\_\_____/\___|\__,_|_|  |_| |_|

        xLearn   -- 0.40 Version --
----------------------------------------------------------------------------------------------

[------------] xLearn uses 2 threads for training task.
[ ACTION     ] Read Problem ...
[------------] First check if the text file has been already converted to binary format.
[------------] Binary file (libffm_train_fold_4.txt.bin) NOT found. Convert text file to binary file.
[------------] First check if the text file has been already converted to binary format.
[------------] Binary file (libffm_val_fold_4.txt.bin) NOT found. Convert text file to binary file.
[------------] Number of Feature: 1548962
[------------] Number of Field: 35
[------------] Time cost for reading probl

Progress: 100%|██████████████████████| 254655/254655 [00:06<00:00, 38345.95it/s]


----------------------------------------------------------------------------------------------
           _
          | |
     __  _| |     ___  __ _ _ __ _ __
     \ \/ / |    / _ \/ _` | '__| '_ \ 
      >  <| |___|  __/ (_| | |  | | | |
     /_/\_\_____/\___|\__,_|_|  |_| |_|

        xLearn   -- 0.40 Version --
----------------------------------------------------------------------------------------------

[------------] xLearn uses 2 threads for training task.
[ ACTION     ] Read Problem ...
[------------] First check if the text file has been already converted to binary format.
[------------] Binary file (libffm_train_fold_5.txt.bin) NOT found. Convert text file to binary file.
[------------] First check if the text file has been already converted to binary format.
[------------] Binary file (libffm_val_fold_5.txt.bin) NOT found. Convert text file to binary file.
[------------] Number of Feature: 1548961
[------------] Number of Field: 35
[------------] Time cost for reading probl

Progress: 100%|██████████████████████| 254655/254655 [00:06<00:00, 38659.68it/s]


----------------------------------------------------------------------------------------------
           _
          | |
     __  _| |     ___  __ _ _ __ _ __
     \ \/ / |    / _ \/ _` | '__| '_ \ 
      >  <| |___|  __/ (_| | |  | | | |
     /_/\_\_____/\___|\__,_|_|  |_| |_|

        xLearn   -- 0.40 Version --
----------------------------------------------------------------------------------------------

[------------] xLearn uses 2 threads for training task.
[ ACTION     ] Read Problem ...
[------------] First check if the text file has been already converted to binary format.
[------------] Binary file (libffm_train_fold_6.txt.bin) NOT found. Convert text file to binary file.
[------------] First check if the text file has been already converted to binary format.
[------------] Binary file (libffm_val_fold_6.txt.bin) NOT found. Convert text file to binary file.
[------------] Number of Feature: 1548962
[------------] Number of Field: 35
[------------] Time cost for reading probl

## Evaluation and Submission

In [9]:
oof_df = pd.DataFrame({ID: train_id, TARGET: oof_preds})
oof_df.to_csv('oof_preds.csv', index=False)

sub = pd.DataFrame({ID: test_id, TARGET: test_preds})
sub.to_csv('submission.csv', index=False)
sub.head()

,id,Churn
0,594194,-2.371594
1,594195,-6.201360
2,594196,-2.127626
3,594197,-5.136244
4,594198,0.100693
